# 01. 순서 데이터와 RNNCell

In [1]:
import torch                                       # 파이토치
from torch import nn                               # 신경망 모듈

cell = nn.RNNCell(1, 1).double()                   # 입력 1개 → 상태 1개, float64
with torch.no_grad():                              # 값 설정은 미분 기록에 넣지 않음
    cell.weight_ih.fill_(0.6)                      # W_ih = 0.6 (모든 원소를 채움)
    cell.weight_hh.fill_(0.5)                      # W_hh = 0.5
    cell.bias_ih.fill_(0.1)                        # b_ih = 0.1
    cell.bias_hh.fill_(0.0)                        # b_hh = 0

h = torch.zeros(1, 1, dtype=torch.float64)         # h0 = 0, 모양 (예제 수, 상태 수)
for x in [2.0, 1.0, -1.0]:                         # 시점 순서대로 입력
    xt = torch.tensor([[x]], dtype=torch.float64)  # 모양 (1, 1) 텐서로 변환
    h = cell(xt, h)                                # (현재 입력, 직전 상태) → 새 상태
    print(round(h.item(), 6))                      # 상태 값 하나를 꺼내 출력

0.861723
0.811314
-0.094064


# 02. nn.RNN의 입력과 반환값

In [2]:
import torch                                              # 파이토치
from torch import nn                                      # 신경망 모듈

torch.manual_seed(0)                                      # 무작위 초기값 고정
cell = nn.RNNCell(1, 2)                                   # 입력 1칸, 상태 2칸
seq = torch.tensor([[2.0], [0.0], [-1.0]])                # 시점 3개 × 입력 1칸, 모양 (3, 1)
h = torch.zeros(1, 2)                                     # h0 = 0, 모양 (B, H)
for xt in seq:                                            # 시점 하나씩 꺼냄
    h = cell(xt.reshape(1, 1), h)                         # (현재 입력, 직전 상태) → 새 상태
    print(h.shape)                                        # 상태의 모양

rnn = nn.RNN(1, 2, batch_first=True)                      # 입력 1칸, 상태 2칸, 입력 모양 (B, T, F)
x3 = seq.reshape(1, 3, 1)                                 # (3, 1) → (1, 3, 1)
output, h_n = rnn(x3)                                     # 모든 시점을 한 번에 계산
print(output.shape, h_n.shape)                            # 모든 시점 상태, 마지막 상태

last = output[:, -1, :]                                   # 모든 예제, 마지막 시점, 모든 상태 칸
print(last.shape)                                         # 시간 축이 사라진 모양

x_b2 = torch.tensor([2., 0., -1., 1., 3., 0.]).reshape(2, 3, 1)  # 예제 2 × 시점 3 × 입력 1
rnn_h4 = nn.RNN(1, 4, batch_first=True)                   # 상태 4칸
out4, hn4 = rnn_h4(x_b2)                                  # H=4 결과
print(out4.shape, hn4.shape)                              # 마지막 축이 4

torch.Size([1, 2])
torch.Size([1, 2])
torch.Size([1, 2])
torch.Size([1, 3, 2]) torch.Size([1, 1, 2])
torch.Size([1, 2])
torch.Size([2, 3, 4]) torch.Size([1, 2, 4])


# 03. RNN으로 시계열 예측

In [3]:
import torch                                                         # 파이토치
from torch import nn                                                 # 신경망 모듈

data = torch.tensor([5., 7., 6., 9., 8., 11., 10., 13., 12., 15.])   # 10일치, 모양 (10,)
L = 4                                                                # 입력창 길이
X = torch.stack([data[i:i+L] for i in range(len(data) - L)])         # 창 6개를 쌓음, (6, 4)
X = X.unsqueeze(-1)                                                  # 끝에 크기 1인 축 추가, (6, 4, 1)
y = data[L:].unsqueeze(-1)                                           # 정답, (6, 1)

class SimpleRNN(nn.Module):                                          # 모델 클래스
    def __init__(self, hidden_size):                                 # 층을 만드는 곳
        super().__init__()                                           # 부모 클래스 준비
        self.rnn = nn.RNN(1, hidden_size, batch_first=True)          # 입력 1칸, 상태 H칸
        self.fc = nn.Linear(hidden_size, 1)                          # 상태 H칸 → 예측 1칸

    def forward(self, x):                                            # 계산 순서를 정하는 곳
        output, h_n = self.rnn(x)                                    # 모든 시점의 상태 (B, T, H)
        last = output[:, -1, :]                                      # 마지막 시점의 상태 (B, H)
        return self.fc(last)                                         # 예측 (B, 1)

model = SimpleRNN(8)                                                 # 상태 8칸인 모델
pred = model(X)                                                      # forward 실행, 예측 (6, 1)
print(X.shape, y.shape, pred.shape)                                  # 세 모양 확인

torch.Size([6, 4, 1]) torch.Size([6, 1]) torch.Size([6, 1])


# 04. 손실과 BPTT

In [1]:
import torch                                                    # 파이토치
from torch import nn                                            # 신경망 모듈

rnn = nn.RNN(1, 1, bias=False, batch_first=True).double()       # 입력 1칸, 상태 1칸, 편향 없음
with torch.no_grad():                                           # 값 설정은 미분 기록에 넣지 않음
    rnn.weight_ih_l0.fill_(0.5)                                 # W_ih = 0.5
    rnn.weight_hh_l0.fill_(0.4)                                 # W_hh = 0.4

x = torch.tensor([[[2.0], [0.0], [1.0]]], dtype=torch.float64)  # 입력, 모양 (1, 3, 1)
y = torch.tensor([[0.3]], dtype=torch.float64)                  # 정답, 모양 (1, 1)

output, h_n = rnn(x)                                            # 모든 시점의 상태
pred = output[:, -1, :]                                         # 마지막 상태를 예측으로 사용
loss = 0.5 * ((pred - y) ** 2).sum()                            # 손실 L = ½(예측 − 정답)²
loss.backward()                                                 # 기울기 계산 (가중치는 바뀌지 않음)

print(round(loss.item(), 6))                                    # 손실 값
print(round(rnn.weight_hh_l0.grad.item(), 6), round(rnn.weight_ih_l0.grad.item(), 6))  # W_hh, W_ih의 기울기

0.031222
0.099991 0.195709
